# Backtesting Assignment (Jitta)

This notebook simulates the trading rules in the assignment using Close prices and 100-share lots.

Assumptions
- If a Buy Date is not a trading day, use the next available trading date in the price data.
- Missing close prices for a held stock are filled with the last available close (forward-fill).
- Fees are accrued only on trading days present in the dataset.
- Rebalance runs before the daily fee; on Dec 31 the accrued fee is paid after that day's fee is accrued.


In [6]:
import pandas as pd
import numpy as np

DATA_PATH = "interday (intel-assignment).csv"
START_DATE = "2018-12-31"
END_DATE = "2021-12-31"

INITIAL_CASH = 100_000.0
LOT_SIZE = 100

ANNUAL_FEE_RATE = 0.0005  # 0.05%
DAILY_FEE_RATE = ANNUAL_FEE_RATE / 365


In [7]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df[(df["timestamp"] >= START_DATE) & (df["timestamp"] <= END_DATE)]

prices = df.pivot_table(index="timestamp", columns="jitta_stock_id", values="close")
prices = prices.sort_index()
prices = prices.ffill()  # carry forward last close on missing days

prices.head()


jitta_stock_id,3695,4572,7634,8893,497280,851607,1232815,2572065,2572066,2572067,2572286,2572765,2572856,2573042,2573062,2573085,2573125,2577965,2583162
timestamp,,,,,,,,,,,,,,,,,,,
2018-12-31,20.952306,7.998225,15.845512,32.126911,NaN,NaN,84.939420,54.367336,29.857637,19.442551,39.177387,32.883053,NaN,16.444390,15.056340,26.239624,22.384640,NaN,NaN
2019-01-02,20.999730,8.056324,15.850127,32.155889,NaN,NaN,85.562258,53.733930,30.173486,19.548584,39.207240,32.697659,NaN,16.335140,14.849654,26.336880,23.712999,NaN,NaN
2019-01-03,20.202993,7.988542,15.236825,31.305869,NaN,NaN,80.423851,52.975763,29.490857,18.862455,37.734481,31.819477,NaN,15.927514,14.331890,25.539382,23.730950,NaN,NaN
2019-01-04,21.170460,8.230619,15.825877,32.378053,NaN,NaN,84.170606,54.501695,31.061813,19.693174,39.615234,32.902568,NaN,16.584708,14.900278,26.511940,24.314351,NaN,NaN
2019-01-07,21.483464,8.298401,16.179309,32.870679,NaN,NaN,85.795822,54.655248,32.078453,20.184780,40.381467,33.624629,NaN,16.584708,14.811947,26.988494,24.987506,NaN,NaN


In [8]:
rank_schedule = {
    "2018-12-31": [2573042, 2572286, 1232815, 2572066],
    "2019-03-31": [2573125, 3695, 2572066, 2572067],
    "2019-06-30": [3695, 8893, 2572065, 2572067],
    "2019-09-30": [2573062, 2572286, 3695, 2572067],
    "2019-12-31": [2572856, 2572065, 851607, 2572765],
    "2020-03-31": [2573062, 851607, 497280, 2572066],
    "2020-06-30": [851607, 2572066, 497280, 2572067],
    "2020-09-30": [4572, 851607, 2572066, 1232815],
    "2020-12-31": [4572, 2572067, 2572765, 3695],
    "2021-03-31": [1232815, 2572067, 2572065, 7634],
    "2021-06-30": [2573085, 2572065, 2573062, 497280],
    "2021-09-30": [7634, 497280, 3695, 1232815],
}

price_dates = prices.index

def map_to_trading_date(raw_date):
    if raw_date in price_dates:
        return raw_date
    pos = price_dates.searchsorted(raw_date)
    if pos >= len(price_dates):
        raise ValueError(f"No trading date on or after {raw_date}")
    return price_dates[pos]

schedule_rows = []
effective_schedule = {}

for raw_date, stocks in rank_schedule.items():
    raw_dt = pd.to_datetime(raw_date)
    eff_dt = map_to_trading_date(raw_dt)

    missing_ids = [sid for sid in stocks if sid not in prices.columns]
    if missing_ids:
        raise ValueError(f"Missing stock ids in price data: {missing_ids}")

    if eff_dt in effective_schedule:
        raise ValueError(f"Duplicate effective date: {eff_dt}")

    effective_schedule[eff_dt] = stocks
    schedule_rows.append({
        "buy_date": raw_dt.date(),
        "effective_date": eff_dt.date(),
        "stocks": stocks,
    })

schedule_df = pd.DataFrame(schedule_rows).sort_values("effective_date")
schedule_df


,buy_date,effective_date,stocks
0,2018-12-31,2018-12-31,"[2573042, 2572286, 1232815, 2572066]"
1,2019-03-31,2019-04-01,"[2573125, 3695, 2572066, 2572067]"
2,2019-06-30,2019-07-01,"[3695, 8893, 2572065, 2572067]"
3,2019-09-30,2019-09-30,"[2573062, 2572286, 3695, 2572067]"
4,2019-12-31,2019-12-31,"[2572856, 2572065, 851607, 2572765]"
5,2020-03-31,2020-03-31,"[2573062, 851607, 497280, 2572066]"
6,2020-06-30,2020-06-30,"[851607, 2572066, 497280, 2572067]"
7,2020-09-30,2020-09-30,"[4572, 851607, 2572066, 1232815]"
8,2020-12-31,2020-12-31,"[4572, 2572067, 2572765, 3695]"
9,2021-03-31,2021-03-31,"[1232815, 2572067, 2572065, 7634]"


In [9]:
def get_price_row(date):
    try:
        row = prices.loc[date]
    except KeyError as exc:
        raise ValueError(f"Missing price data for {date}") from exc
    return row


def rebalance(date, target_stocks, holdings, cash, tx_log):
    price_row = get_price_row(date)

    if price_row[target_stocks].isna().any():
        missing = price_row[target_stocks][price_row[target_stocks].isna()].index.tolist()
        raise ValueError(f"Missing price for target stocks on {date}: {missing}")

    total_value = cash
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        px = price_row.get(sid)
        if pd.isna(px):
            raise ValueError(f"Missing price for holding {sid} on {date}")
        total_value += shares * px

    target_value = total_value / len(target_stocks)

    desired = {}
    for sid in target_stocks:
        px = price_row[sid]
        lots = np.floor(target_value / (px * LOT_SIZE))
        desired[sid] = int(lots * LOT_SIZE)

    all_sids = sorted(set(holdings.keys()) | set(target_stocks))
    for sid in all_sids:
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur > tgt:
            shares = cur - tgt
            px = price_row[sid]
            proceeds = shares * px
            cash += proceeds
            holdings[sid] = tgt
            tx_log.append({
                "date": date,
                "action": "SELL",
                "stock_id": sid,
                "shares": shares,
                "price": px,
                "amount": proceeds,
                "note": "rebalance",
            })

    for sid in sorted(target_stocks):
        cur = holdings.get(sid, 0)
        tgt = desired.get(sid, 0)
        if cur < tgt:
            shares = tgt - cur
            px = price_row[sid]
            cost = shares * px
            if cost > cash + 1e-9:
                max_lots = int(cash // (px * LOT_SIZE))
                shares = max_lots * LOT_SIZE
                if shares == 0:
                    continue
                cost = shares * px
                tgt = cur + shares
            cash -= cost
            holdings[sid] = tgt
            tx_log.append({
                "date": date,
                "action": "BUY",
                "stock_id": sid,
                "shares": shares,
                "price": px,
                "amount": -cost,
                "note": "rebalance",
            })

    return holdings, cash


In [10]:
dates = prices.index[(prices.index >= START_DATE) & (prices.index <= END_DATE)]

holdings = {}
cash = INITIAL_CASH
accrued_fee = 0.0

tx_log = []
daily_rows = []

for date in dates:
    if date in effective_schedule:
        holdings, cash = rebalance(date, effective_schedule[date], holdings, cash, tx_log)

    price_row = prices.loc[date]
    stock_value = 0.0
    for sid, shares in holdings.items():
        if shares == 0:
            continue
        px = price_row[sid]
        if pd.isna(px):
            raise ValueError(f"Missing price for holding {sid} on {date}")
        stock_value += shares * px

    asset_value = stock_value + cash - accrued_fee
    daily_fee = asset_value * DAILY_FEE_RATE
    accrued_fee += daily_fee

    if date.month == 12 and date.day == 31 and accrued_fee > 0:
        cash -= accrued_fee
        tx_log.append({
            "date": date,
            "action": "FEE_PAYMENT",
            "stock_id": None,
            "shares": None,
            "price": None,
            "amount": -accrued_fee,
            "note": "year-end",
        })
        accrued_fee = 0.0

    nav = stock_value + cash - accrued_fee

    daily_rows.append({
        "date": date,
        "stock_value": stock_value,
        "cash": cash,
        "accrued_fee": accrued_fee,
        "daily_fee": daily_fee,
        "nav": nav,
    })

tx_df = pd.DataFrame(tx_log).sort_values("date").reset_index(drop=True)
daily_nav = pd.DataFrame(daily_rows).set_index("date")

tx_df.head()


,date,action,stock_id,shares,price,amount,note
0,2018-12-31,BUY,1232815.0,200.0,84.939420,-16987.884000,rebalance
1,2018-12-31,BUY,2572066.0,800.0,29.857637,-23886.109600,rebalance
2,2018-12-31,BUY,2572286.0,600.0,39.177387,-23506.432200,rebalance
3,2018-12-31,BUY,2573042.0,1500.0,16.444390,-24666.585000,rebalance
4,2018-12-31,FEE_PAYMENT,NaN,NaN,NaN,-0.136986,year-end


In [11]:
tx_df


,date,action,stock_id,shares,price,amount,note
0,2018-12-31,BUY,1232815.0,200.0,84.939420,-16987.884000,rebalance
1,2018-12-31,BUY,2572066.0,800.0,29.857637,-23886.109600,rebalance
2,2018-12-31,BUY,2572286.0,600.0,39.177387,-23506.432200,rebalance
3,2018-12-31,BUY,2573042.0,1500.0,16.444390,-24666.585000,rebalance
4,2018-12-31,FEE_PAYMENT,NaN,NaN,NaN,-0.136986,year-end
...,...,...,...,...,...,...,...
71,2021-09-30,SELL,2573062.0,1600.0,32.716496,52346.393600,rebalance
72,2021-09-30,SELL,2573085.0,800.0,81.816769,65453.415200,rebalance
73,2021-09-30,BUY,3695.0,1400.0,45.546841,-63765.577400,rebalance
74,2021-09-30,BUY,7634.0,1700.0,36.192686,-61527.566200,rebalance


In [12]:
daily_nav


,stock_value,cash,accrued_fee,daily_fee,nav
date,,,,,
2018-12-31,89047.0108,10952.852214,0.000000,0.136986,99999.863014
2019-01-02,89278.2944,10952.852214,0.137303,0.137303,100231.009311
2019-01-03,86209.4154,10952.852214,0.270402,0.133099,97161.997212
2019-01-04,90329.7740,10952.852214,0.409145,0.138743,101282.217069
2019-01-07,91927.8690,10952.852214,0.550077,0.140932,102880.171137
...,...,...,...,...,...
2021-12-27,246325.9450,18881.223981,88.745749,0.363176,265118.423232
2021-12-28,243158.0981,18881.223981,89.104586,0.358836,261950.217496
2021-12-29,242537.4779,18881.223981,89.462571,0.357986,261329.239310


In [13]:
nav_start = daily_nav["nav"].iloc[0]
nav_end = daily_nav["nav"].iloc[-1]
years = (daily_nav.index[-1] - daily_nav.index[0]).days / 365

total_return = (nav_end - nav_start) / nav_start
cagr = (nav_end / nav_start) ** (1 / years) - 1

pd.DataFrame(
    {"value": [nav_start, nav_end, total_return, cagr, years]},
    index=["NAV_start", "NAV_end", "Total_Return", "CAGR", "Years"],
)


,value
NAV_start,99999.863014
NAV_end,260175.045914
Total_Return,1.601754
CAGR,0.374978
Years,3.002740


In [14]:
tx_df

,date,action,stock_id,shares,price,amount,note
0,2018-12-31,BUY,1232815.0,200.0,84.939420,-16987.884000,rebalance
1,2018-12-31,BUY,2572066.0,800.0,29.857637,-23886.109600,rebalance
2,2018-12-31,BUY,2572286.0,600.0,39.177387,-23506.432200,rebalance
3,2018-12-31,BUY,2573042.0,1500.0,16.444390,-24666.585000,rebalance
4,2018-12-31,FEE_PAYMENT,NaN,NaN,NaN,-0.136986,year-end
...,...,...,...,...,...,...,...
71,2021-09-30,SELL,2573062.0,1600.0,32.716496,52346.393600,rebalance
72,2021-09-30,SELL,2573085.0,800.0,81.816769,65453.415200,rebalance
73,2021-09-30,BUY,3695.0,1400.0,45.546841,-63765.577400,rebalance
74,2021-09-30,BUY,7634.0,1700.0,36.192686,-61527.566200,rebalance
